### MLOps AIMLCZG523
### Assignment 1

#### Heart Disease Risk Prediction: End-to-End ML Pipeline


|Name|BITS ID|
|---|---|
|Devang Ajay Kamble | 2025AE05173 |

#### Pre-requisites

We would be using Python 3.10 and the following libraries:
- numpy==2.2.6
- pandas==2.3.3
- scikit-learn==1.7.2
- matplotlib==3.10.9
- seaborn==0.13.2
- mlflow==3.17.1
- joblib==1.6.0
- requests==2.34.2


#### Imports and configuration

The reusable logic lives in `src/` (`data.py`, `features.py`, `train.py`) so it can be unit-tested and reused by the API later. This notebook runs it step by step and prints the key functions next to their explanation.

In [1]:
import inspect
import json
import warnings

import joblib
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import seaborn as sns

from src import data, features, train
from src.config import (CATEGORICAL, EXPERIMENT, FEATURES, FIG_DIR, LABELS, MODEL_DIR,
                        NUMERIC, ROOT, SEED, TEST_SIZE)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")

2026/10/17 10:50:53 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\marcus\AppData\Local\Temp\claude\c--Users-marcus-Desktop-MTech-MLOps-Assignment-1\5147d50f-af78-4804-814f-14da312ba812\scratchpad\venv\lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


#### Task-1 - Data Acquisition & Exploratory Data Analysis

Source: UCI Machine Learning Repository, Heart Disease dataset. We download the dataset and then use locally

In [2]:
print(inspect.getsource(data.download))

def download(force=False):
    """Download the raw Cleveland file once; skip if it already exists."""
    if config.RAW_PATH.exists() and not force:
        return config.RAW_PATH
    config.RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    resp = requests.get(config.DATA_URL, timeout=30)
    resp.raise_for_status()
    config.RAW_PATH.write_bytes(resp.content)
    return config.RAW_PATH



In [3]:
raw_path = data.download()
raw = data.load_raw()

print("Saved to :", raw_path.relative_to(ROOT))
print("Shape    :", raw.shape)
raw.head()

Saved to : data\raw\processed.cleveland.data
Shape    : (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


#### Data dictionary

| Feature | Meaning | Type |
|---|---|---|
| `age` | age in years | numeric |
| `sex` | 1 = male, 0 = female | binary |
| `cp` | chest pain type (1 typical angina, 2 atypical, 3 non-anginal, 4 asymptomatic) | nominal |
| `trestbps` | resting blood pressure (mm Hg) | numeric |
| `chol` | serum cholesterol (mg/dl) | numeric |
| `fbs` | fasting blood sugar > 120 mg/dl | binary |
| `restecg` | resting ECG result (0 normal, 1 ST-T abnormality, 2 LV hypertrophy) | nominal |
| `thalach` | maximum heart rate achieved | numeric |
| `exang` | exercise-induced angina | binary |
| `oldpeak` | ST depression induced by exercise | numeric |
| `slope` | slope of the peak exercise ST segment (1 up, 2 flat, 3 down) | nominal |
| `ca` | major vessels coloured by fluoroscopy (0-3) | ordinal |
| `thal` | thalassemia (3 normal, 6 fixed defect, 7 reversible defect) | nominal |
| `target` | 0 = no disease, 1-4 = disease severity | label |

#### Data quality checks

In [4]:
raw_missing = (raw == "?").sum()
print("'?' placeholders per column:")
print(raw_missing[raw_missing > 0].to_dict())
print("\nDuplicate rows:", raw.duplicated().sum())
print("\nRaw target values:", raw["target"].value_counts().sort_index().to_dict())

'?' placeholders per column:
{'ca': 4, 'thal': 2}

Duplicate rows: 0

Raw target values: {0: 164, 1: 55, 2: 36, 3: 35, 4: 13}


#### Cleaning

* `?` in `ca` / `thal` becomes `NaN`; all columns are cast to numeric.
* Target is binarised: 0 = no disease, 1-4 = disease.
* Missing values are imputed later, inside the model pipeline, so the imputer learns from training folds only.

The function is printed below (it lives in `src/data.py`).

In [5]:
print(inspect.getsource(data.clean))

def clean(raw):
    """'?' -> NaN, numeric types, drop duplicates, binarise the target.

    Missing values stay NaN here and are imputed inside the model pipeline,
    so the imputer is fit on training folds only (no leakage).
    """
    df = raw.replace("?", pd.NA).apply(pd.to_numeric, errors="coerce")
    df = df.drop_duplicates().reset_index(drop=True)
    # original target: 0 = healthy, 1-4 = disease severity -> binary
    df[config.TARGET] = (df[config.TARGET] > 0).astype(int)
    # float features keep the MLflow model signature stable when NaN appears
    feats = [c for c in df.columns if c != config.TARGET]
    df[feats] = df[feats].astype("float64")
    return df



In [6]:
df = data.build_clean_dataset()   # clean + save to data/processed/

print("Saved to: data/processed/heart_clean.csv", df.shape)
print("\nMissing values after cleaning:", df.isna().sum()[lambda s: s > 0].to_dict())
df.describe().T.round(2)

Saved to: data/processed/heart_clean.csv (303, 14)

Missing values after cleaning: {'ca': 4, 'thal': 2}


,count,mean,std,min,25%,50%,75%,max
age,303.0,54.44,9.04,29.0,48.0,56.0,61.0,77.0
sex,303.0,0.68,0.47,0.0,0.0,1.0,1.0,1.0
cp,303.0,3.16,0.96,1.0,3.0,3.0,4.0,4.0
trestbps,303.0,131.69,17.60,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.69,51.78,126.0,211.0,241.0,275.0,564.0
fbs,303.0,0.15,0.36,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.99,0.99,0.0,0.0,1.0,2.0,2.0
thalach,303.0,149.61,22.88,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.33,0.47,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.04,1.16,0.0,0.0,0.8,1.6,6.2


Expected Result

* 303 patients, 14 columns, no duplicates.
* Only 6 missing cells (`ca`: 4, `thal`: 2), so imputing them costs almost nothing.
* `trestbps` (max 200), `chol` (max 564) and `oldpeak` (max 6.2) have long right tails.

#### Class balance

In [7]:
counts = df["target"].map(LABELS).value_counts()

fig, ax = plt.subplots(figsize=(5, 3.6))
sns.barplot(x=counts.index, y=counts.values, ax=ax)
for i, v in enumerate(counts.values):
    ax.text(i, v + 2, f"{v} ({v / len(df):.0%})", ha="center")
ax.set_title("Class balance")
ax.set_ylabel("patients")
ax.set_xlabel("")
fig.savefig(FIG_DIR / "eda_class_balance.png", dpi=130, bbox_inches="tight")
plt.show()

The classes are close to balanced (54% / 46%), so no resampling is needed. Accuracy is a fair metric here, but ROC-AUC, precision and recall are still reported because missing a sick patient (recall) and flagging a healthy one (precision) cost very different things.

#### Numeric features: distributions by class

In [8]:
numeric = NUMERIC
plot_df = df.assign(label=df["target"].map(LABELS))

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.ravel(), numeric):
    sns.histplot(data=plot_df, x=col, hue="label", kde=True, bins=20, ax=ax, stat="density", common_norm=False)
    ax.set_title(col)
axes.ravel()[-1].axis("off")
fig.suptitle("Numeric feature distributions by class", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_numeric_hist.png", dpi=130, bbox_inches="tight")
plt.show()

Expected Result

* `thalach` is lower and `oldpeak` higher for patients with disease: the best numeric separators.
* `chol` and `trestbps` overlap heavily between classes: weak predictors.

#### Outliers

In [9]:
fig, axes = plt.subplots(1, 5, figsize=(14, 3.5))
for ax, col in zip(axes, numeric):
    sns.boxplot(data=plot_df, x="label", y=col, ax=ax)
    ax.set_xlabel("")
    ax.set_title(col)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_boxplots.png", dpi=130, bbox_inches="tight")
plt.show()

# IQR rule, to count not to drop
q1, q3 = df[numeric].quantile(0.25), df[numeric].quantile(0.75)
iqr = q3 - q1
outliers = ((df[numeric] < q1 - 1.5 * iqr) | (df[numeric] > q3 + 1.5 * iqr)).sum()
print("IQR outliers per column:", outliers.to_dict())

IQR outliers per column: {'age': 0, 'trestbps': 9, 'chol': 5, 'thalach': 1, 'oldpeak': 5}


The flagged values (cholesterol of 564 mg/dl, resting BP of 200) are extreme but clinically plausible, and with only 303 rows dropping them would throw away real patients. They are kept and handled with `StandardScaler`, and the tree models are insensitive to them anyway.

#### Categorical features: disease rate per category

In [10]:
cats = CATEGORICAL + ["sex", "fbs", "exang", "ca"]

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, col in zip(axes.ravel(), cats):
    rate = df.groupby(col)["target"].mean()
    n = df.groupby(col).size()
    sns.barplot(x=rate.index.astype(int), y=rate.values, ax=ax, color="#3b6ea5")
    for i, (r, c) in enumerate(zip(rate.values, n.values)):
        ax.text(i, r + 0.01, f"n={c}", ha="center", fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_title(col)
    ax.set_ylabel("disease rate")
    ax.set_xlabel("")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_categorical_rates.png", dpi=130, bbox_inches="tight")
plt.show()

Expected Result

* Disease rate is high for `cp = 4` (73%), `thal` 6/7 (67-76%), `exang = 1` (77%) and rises with `ca`; males 55% vs females 26%.
* `fbs` barely moves the rate.

#### Correlation heatmap

In [11]:
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True,
            linewidths=0.5, cbar_kws={"shrink": 0.7}, ax=ax)
ax.set_title("Correlation matrix (Pearson)")
fig.savefig(FIG_DIR / "eda_correlation.png", dpi=130, bbox_inches="tight")
plt.show()

corr["target"].drop("target").sort_values(key=abs, ascending=False).round(2)

thal        0.53
ca          0.46
exang       0.43
oldpeak     0.42
thalach    -0.42
cp          0.41
slope       0.34
sex         0.28
age         0.22
restecg     0.17
trestbps    0.15
chol        0.09
fbs         0.03
Name: target, dtype: float64

Expected Result

* Strongest links to the target: `thal` (0.53), `ca` (0.46), `exang`, `oldpeak`, `cp` (about 0.42) and `thalach` (-0.42).
* `chol`, `fbs` and `restecg` are almost uncorrelated with it; no feature pair is highly collinear.

### Task-2: Feature Engineering & Model Development

#### Train / test split

20% of patients are held out and used once, at the end. Tuning and model choice use cross-validation on the other 80%. The split is stratified to keep the class ratio.

In [12]:
X_tr, X_te, y_tr, y_te = train.split(df)

print("train:", X_tr.shape, y_tr.mean().round(3), "| test:", X_te.shape, y_te.mean().round(3))

train: (242, 13) 0.459 | test: (61, 13) 0.459


#### Pipeline design

All learning steps sit in one `Pipeline`, so cross-validation never leaks validation data:

```
raw 13 features -> FeatureEngineer -> impute / scale / one-hot -> SelectKBest(k) -> classifier
```

Instead of hand-picking variables, two choices are tuned by cross-validated ROC-AUC:

* `fe__add_features`: include `hr_pct_max = thalach / (220 - age)` and `oldpeak_x_exang` or not.
* `select__k`: keep 8, 12, 16 or all of the 22-24 encoded columns (ranked by ANOVA F-score).

In [13]:
print(inspect.getsource(features.FeatureEngineer))
print(inspect.getsource(features.build_pipeline))

class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Adds two clinically motivated features; switchable so CV can judge them."""

    def __init__(self, add_features=True):
        self.add_features = add_features

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        if self.add_features:
            X["hr_pct_max"] = X["thalach"] / (220.0 - X["age"])
            X["oldpeak_x_exang"] = X["oldpeak"] * X["exang"]
        return X

def build_pipeline(estimator):
    """engineer -> impute/scale/one-hot -> select top-k -> classifier."""
    numeric = Pipeline([("impute", SimpleImputer(strategy="median")),
                        ("scale", StandardScaler())])
    nominal = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    ordinal = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
       

#### Candidate models

Logistic Regression (strong baseline for small tabular data), Random Forest (non-linear, low variance) and Gradient Boosting (usually strong on tabular data), each tuned with a full grid.

In [14]:
print(inspect.getsource(train.candidates))
print("SHARED_GRID =", train.SHARED_GRID, "\n")

for name, (est, grid) in train.candidates().items():
    n = 1
    for v in {**train.SHARED_GRID, **grid}.values():
        n *= len(v)
    print(f"{name:20s} {n:4d} configurations x 5 folds")

def candidates():
    """name -> (estimator, model-specific grid). Seeds fixed for reproducibility."""
    return {
        "logistic_regression": (
            LogisticRegression(max_iter=2000, random_state=config.SEED),
            {"clf__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10],
             "clf__class_weight": [None, "balanced"]},
        ),
        "random_forest": (
            RandomForestClassifier(n_estimators=300, random_state=config.SEED, n_jobs=1),
            {"clf__max_depth": [3, 5, None],
             "clf__min_samples_leaf": [1, 3, 5],
             "clf__max_features": ["sqrt", 0.5]},
        ),
        "gradient_boosting": (
            GradientBoostingClassifier(random_state=config.SEED),
            {"clf__n_estimators": [50, 100, 200],
             "clf__learning_rate": [0.03, 0.1],
             "clf__max_depth": [2, 3],
             "clf__subsample": [0.8, 1.0]},
        ),
    }

SHARED_GRID = {'fe__add_features': [False, True], 'select__k': [8, 12, 16, 'all']} 

lo

#### Evaluation protocol

* **Tuning:** 5-fold stratified `GridSearchCV` on ROC-AUC.
* **Comparison:** nested CV, so the score is not inflated by the search. The winner is picked on this, never on the test set.
* **Metrics:** accuracy, precision, recall, F1, ROC-AUC.

In [15]:
print(inspect.getsource(train.make_search))
print(inspect.getsource(train.nested_cv))

def make_search(estimator, grid, inner_splits=5):
    """Grid search over the whole pipeline, so selection + FE live inside CV."""
    cv = StratifiedKFold(inner_splits, shuffle=True, random_state=config.SEED)
    return GridSearchCV(features.build_pipeline(estimator), {**SHARED_GRID, **grid},
                        scoring="roc_auc", cv=cv, n_jobs=-1, refit=True)

def nested_cv(estimator, grid, X, y, outer_splits=5, inner_splits=3):
    """Tuning is repeated inside every outer training fold -> unbiased estimate."""
    outer = StratifiedKFold(outer_splits, shuffle=True, random_state=config.SEED + 1)
    res = cross_validate(make_search(estimator, grid, inner_splits), X, y,
                         cv=outer, scoring=SCORING)
    return {k.replace("test_", ""): (v.mean(), v.std())
            for k, v in res.items() if k.startswith("test_")}



### Task-3: Experiment Tracking

MLflow records one parent run (`model-selection`) and one child run per model: parameters, metrics, plots, the tuning table and the fitted pipeline.

In [16]:
train.setup_mlflow()

In [17]:
print(inspect.getsource(train.run_model))

def run_model(name, estimator, grid, X_tr, X_te, y_tr, y_te):
    """One MLflow run per candidate: tune, nested-CV, test, log everything."""
    with mlflow.start_run(run_name=name, nested=True):
        search = make_search(estimator, grid)
        search.fit(X_tr, y_tr)
        best = search.best_estimator_
        cv = nested_cv(estimator, grid, X_tr, y_tr)
        test = test_metrics(best, X_te, y_te)
        kept = features.selected_features(best)

        mlflow.log_params({
            "model": name, "seed": config.SEED, "test_size": config.TEST_SIZE,
            "outer_cv": "5-fold stratified", "inner_cv": "5-fold stratified (3 inside nested)",
            "n_candidates": len(search.cv_results_["params"]), "n_features_kept": len(kept),
            **{k: str(v) for k, v in search.best_params_.items()},
        })
        mlflow.log_metric("tuning_best_cv_roc_auc", search.best_score_)
        for m, (mu, sd) in cv.items():
            mlflow.log_metric(f"cv_{m}", mu)
            

Running the cell below tunes and logs all three models (about 4 minutes).

In [18]:
results, summary, winner = train.train_all(X_tr, X_te, y_tr, y_te)

print("winner:", winner["model"])

[logistic_regression] tuning ...


2026/10/17 10:51:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[random_forest] tuning ...


2026/10/17 10:53:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[gradient_boosting] tuning ...


2026/10/17 10:54:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


winner: logistic_regression


#### Model comparison

In [19]:
summary.round(3).set_index("model")

,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
model,,,,,,,,,,
logistic_regression,0.797,0.788,0.764,0.775,0.891,0.852,0.788,0.929,0.852,0.951
random_forest,0.797,0.795,0.747,0.769,0.878,0.902,0.867,0.929,0.897,0.945
gradient_boosting,0.793,0.798,0.747,0.770,0.861,0.869,0.833,0.893,0.862,0.945


In [20]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cv_cols = ["cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc"]
te_cols = [c.replace("cv_", "test_") for c in cv_cols]
summary.set_index("model")[cv_cols].plot.bar(ax=axes[0], rot=15, ylim=(0.6, 1.0), title="Nested CV (training set)")
summary.set_index("model")[te_cols].plot.bar(ax=axes[1], rot=15, ylim=(0.6, 1.0), title="Held-out test set")
for ax in axes:
    ax.legend(fontsize=7, loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "model_comparison.png", dpi=130, bbox_inches="tight")
plt.show()

#### Which variables did the search pick?

In [21]:
def tuning_table(r):
    path = mlflow.artifacts.download_artifacts(run_id=r["run_id"], artifact_path="tuning/cv_results.csv")
    return pd.read_csv(path)

rows = []
for r in results:
    for (fe, k), g in tuning_table(r).groupby(["param_fe__add_features", "param_select__k"]):
        rows.append({"model": r["model"], "engineered": fe, "k": k, "best_cv_auc": g["mean_test_score"].max()})
ablation = pd.DataFrame(rows)

print("Effect of the engineered features (best CV ROC-AUC, any k):")
display(ablation.groupby(["model", "engineered"])["best_cv_auc"].max().unstack().round(4))
print("Effect of the number of kept features (best CV ROC-AUC, any engineered setting):")
display(ablation.groupby(["model", "k"])["best_cv_auc"].max().unstack().round(4))

Effect of the engineered features (best CV ROC-AUC, any k):


engineered,False,True
model,,
gradient_boosting,0.8960,0.8981
logistic_regression,0.9124,0.9135
random_forest,0.9038,0.9056


Effect of the number of kept features (best CV ROC-AUC, any engineered setting):


k,12,16,8,all
model,,,,
gradient_boosting,0.8960,0.8981,0.8918,0.8918
logistic_regression,0.9124,0.9135,0.9024,0.9114
random_forest,0.9038,0.9056,0.8962,0.9020


In [22]:
fig, ax = plt.subplots(figsize=(7, 4))
sweep = ablation.groupby(["model", "k", "engineered"], as_index=False)["best_cv_auc"].max()
sweep["k"] = sweep["k"].replace("all", "24").astype(int)
sns.lineplot(data=sweep, x="k", y="best_cv_auc", hue="model", style="engineered", markers=True, ax=ax)
ax.set_title("Feature-selection sweep (grid-search CV ROC-AUC)")
ax.set_xlabel("features kept (24 = all)")
fig.savefig(FIG_DIR / "feature_selection_sweep.png", dpi=130, bbox_inches="tight")
plt.show()

#### Winner

In [23]:
print("model            :", winner["model"])
print("best params      :", winner["best_params"])
print(f"features kept    : {len(winner['features'])}")
print("                  ", winner["features"])
print(f"nested CV ROC-AUC: {winner['cv_roc_auc']:.4f}")
print(f"test ROC-AUC     : {winner['test_roc_auc']:.4f}")

model            : logistic_regression
best params      : {'clf__C': 0.3, 'clf__class_weight': 'balanced', 'fe__add_features': True, 'select__k': 16}
features kept    : 16
                   ['age', 'thalach', 'oldpeak', 'hr_pct_max', 'oldpeak_x_exang', 'cp_2.0', 'cp_3.0', 'cp_4.0', 'restecg_2.0', 'slope_1.0', 'slope_2.0', 'thal_3.0', 'thal_7.0', 'sex', 'exang', 'ca']
nested CV ROC-AUC: 0.8907
test ROC-AUC     : 0.9513


In [24]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, suffix in zip(axes, ["roc", "confusion", "importance"]):
    p = mlflow.artifacts.download_artifacts(run_id=winner["run_id"], artifact_path=f"plots/{winner['model']}_{suffix}.png")
    ax.imshow(mpimg.imread(p))
    ax.axis("off")
fig.tight_layout()
plt.show()

Expected Result

* **Winner:** Logistic Regression, nested-CV ROC-AUC 0.891, ahead of Random Forest (0.878) and Gradient Boosting (0.861). The gaps are within fold-to-fold spread, so the simple model is at least as good; with 242 rows the extra capacity of trees does not pay off.
* **Variables:** all models peak at `k = 16`. Keeping 8 costs about 0.01 AUC and keeping all is slightly worse.
* **Engineered features:** add only +0.001 to +0.002 AUC (noise) but never hurt, so they stay.
* **Final model:** `C = 0.3`, balanced class weights, 16 features. Largest weights: `ca`, `cp_4`, `thal_7`, `thal_3`, `sex`.
* **Test set:** 85% accuracy, 0.93 recall, 0.95 ROC-AUC. High recall suits screening. Only 61 test patients, so trust the CV figure more.

#### Why we cleaned, engineered and scaled this way

| Step | What | Why |
|---|---|---|
| Missing `ca`, `thal` | median / most-frequent imputation in the pipeline | only 6 cells, so no rows are lost; no leakage |
| Outliers | kept | extreme but plausible values; 303 rows is too few to discard any |
| Target | 0 vs 1-4 | binary risk problem; per-severity counts too small |
| `cp`, `restecg`, `slope`, `thal` | one-hot | codes have no order (`thal` 3/6/7 is not a scale) |
| `sex`, `fbs`, `exang`, `ca` | kept as numbers | already 0/1 or an ordered count |
| `hr_pct_max`, `oldpeak_x_exang` | engineered | max heart rate means different things by age; ST depression matters more with exertional angina |
| Scaling | `StandardScaler` | Logistic Regression is regularised, so features need comparable scales; min-max would be squashed by outliers like `chol` = 564 |

**Dropped columns:** none by hand. Feature selection removed `trestbps`, `chol` and `fbs` (the weakest in the EDA, correlation 0.15 / 0.09 / 0.03) plus a few categories (`cp_1`, `restecg_0/1`, `slope_3`, `thal_6`). Fewer noisy columns helped on a small dataset: 16 features beat all 24 in CV.

#### Experiment tracking summary

In [25]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT])
runs = runs[runs["tags.mlflow.parentRunId"].notna()]
cols = ["tags.mlflow.runName", "params.n_candidates", "params.n_features_kept", "metrics.tuning_best_cv_roc_auc",
        "metrics.cv_roc_auc", "metrics.cv_roc_auc_std", "metrics.test_accuracy", "metrics.test_precision",
        "metrics.test_recall", "metrics.test_roc_auc"]
runs[cols].rename(columns=lambda c: c.rsplit(".", 1)[-1]).round(3).set_index("runName")

,n_candidates,n_features_kept,tuning_best_cv_roc_auc,cv_roc_auc,cv_roc_auc_std,test_accuracy,test_precision,test_recall,test_roc_auc
runName,,,,,,,,,
gradient_boosting,192,16,0.898,0.861,0.078,0.869,0.833,0.893,0.945
random_forest,144,16,0.906,0.878,0.058,0.902,0.867,0.929,0.945
logistic_regression,112,16,0.914,0.891,0.048,0.852,0.788,0.929,0.951
gradient_boosting,192,16,0.898,0.861,0.078,0.869,0.833,0.893,0.945
random_forest,144,16,0.906,0.878,0.058,0.902,0.867,0.929,0.945
logistic_regression,112,16,0.914,0.891,0.048,0.852,0.788,0.929,0.951


In [26]:
# everything logged for the winning run
client = mlflow.MlflowClient()
run = client.get_run(winner["run_id"])
print("params   :", len(run.data.params))
print("metrics  :", len(run.data.metrics))
print("artifacts:", [a.path for a in client.list_artifacts(winner["run_id"])])
print("\nlaunch the UI with:  mlflow ui --backend-store-uri", mlflow.get_tracking_uri())

params   : 11
metrics  : 16
artifacts: ['plots', 'selected_features.json', 'tuning']

launch the UI with:  mlflow ui --backend-store-uri sqlite:///C:/Users/marcus/Desktop/MTech/MLOps-Assignment-1/mlflow.db


Each child run logs its parameters, 16 metrics, three plots, the tuning table, kept features and the model with its signature. The parent run tags the winner.

### Task-4: Model Packaging & Reproducibility

The winning pipeline is saved as `models/heart_model.joblib` and registered in MLflow as `heart-disease-classifier`. Preprocessing is inside the pipeline, so callers send only the 13 raw features. `models/model_metadata.json` stores parameters, metrics and library versions; dependencies are pinned in `requirements.txt`.

In [27]:
meta = train.package(winner, summary)
print(json.dumps(meta, indent=2))

{
  "model": "logistic_regression",
  "best_params": {
    "clf__C": "0.3",
    "clf__class_weight": "balanced",
    "fe__add_features": "True",
    "select__k": "16"
  },
  "input_features": [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak",
    "cp",
    "restecg",
    "slope",
    "thal",
    "sex",
    "fbs",
    "exang",
    "ca"
  ],
  "selected_features": [
    "age",
    "thalach",
    "oldpeak",
    "hr_pct_max",
    "oldpeak_x_exang",
    "cp_2.0",
    "cp_3.0",
    "cp_4.0",
    "restecg_2.0",
    "slope_1.0",
    "slope_2.0",
    "thal_3.0",
    "thal_7.0",
    "sex",
    "exang",
    "ca"
  ],
  "cv_metrics": {
    "accuracy": 0.7971,
    "precision": 0.7882,
    "recall": 0.7644,
    "f1": 0.7753,
    "roc_auc": 0.8907
  },
  "test_metrics": {
    "accuracy": 0.8525,
    "precision": 0.7879,
    "recall": 0.9286,
    "f1": 0.8525,
    "roc_auc": 0.9513
  },
  "mlflow_run_id": "35dff0c50e9d4835b0850c0e2f467793",
  "versions": {
    "python": "3.10.11",


Registered model 'heart-disease-classifier' already exists. Creating a new version of this model...
Created version '2' of model 'heart-disease-classifier'.


#### Reproducibility checks

Reload both saved artifacts and confirm they reproduce the in-memory test-set probabilities exactly. Then score a raw record with a missing value to show the pipeline imputes it.

In [28]:
pkl_model = joblib.load(MODEL_DIR / "heart_model.joblib")
mlflow_model = mlflow.sklearn.load_model(winner["model_uri"])

p_live = winner["estimator"].predict_proba(X_te)[:, 1]
print("joblib file == in-memory:", np.allclose(p_live, pkl_model.predict_proba(X_te)[:, 1]))
print("mlflow      == in-memory:", np.allclose(p_live, mlflow_model.predict_proba(X_te)[:, 1]))

sample = pd.DataFrame([{"age": 63, "sex": 1, "cp": 4, "trestbps": 145, "chol": 233, "fbs": 1, "restecg": 2,
                        "thalach": 150, "exang": 1, "oldpeak": 2.3, "slope": 3, "ca": np.nan, "thal": 7}])[FEATURES]
print("\nsample with missing 'ca' -> P(disease) =", round(float(pkl_model.predict_proba(sample)[0, 1]), 4))

joblib file == in-memory: True
mlflow      == in-memory: True

sample with missing 'ca' -> P(disease) = 0.8973


Expected Result

* Both reloaded models match the in-memory probabilities exactly.
* A record with a missing value is still scored, since imputation is part of the pipeline.
* Seeds are fixed (`SEED = 42`), so re-running reproduces these numbers.